# Reprodução do SISSA (Liu et al., 2024) na GPU

Reproduz o **SISSA** — monitoramento de *safety* + *security* em SOME/IP com LSTM/CNN/RNN
+ autoatenção residual (7 classes). Roda o código dos autores (jamesnulliu/SISSA) com a
**janela 128** e mostra **matriz de confusão + métricas por classe + curvas ROC**.

**Use uma GPU:** Runtime → Change runtime type → GPU (T4 já basta).


## 0. Setup — clonar repo dos autores, instalar deps e baixar dados


In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# Clona o repositório oficial dos autores
%cd /content
![ -d SISSA ] || git clone https://github.com/jamesnulliu/SISSA.git
%cd /content/SISSA
!pip -q install ruamel.yaml torchinfo concurrent_log_handler gdown py7zr


In [ ]:
# Baixa o dataset oficial (data.7z no Google Drive) e extrai em data/
import os
if not os.path.exists('data/train/data.npy'):
    import gdown, py7zr
    gdown.download(id='1-pl9OOFcTZCuPLRzusjlmkMn7Q-f-0fU', output='data.7z', quiet=False)
    with py7zr.SevenZipFile('data.7z', 'r') as z:
        z.extractall('.')
print('train:', os.path.exists('data/train/data.npy'), '| val:', os.path.exists('data/val/data.npy'))


In [ ]:
# Usa a janela completa de 128 pacotes (fator decisivo: 64->128 elevou 94%->99%)
from ruamel.yaml import YAML
yaml = YAML()
bc = yaml.load(open('config/basic.yml'))
bc['Preprocessor']['window_height'] = 128
yaml.dump(bc, open('config/basic.yml','w'))
print('window_height =', bc['Preprocessor']['window_height'])


## 1. Treino (configs originais dos autores, já com device: cuda)

Treina os modelos LSTM (com e sem atenção). Adicione outros backbones se quiser
(`SISSA-C`, `SISSA-C-A`, `SISSA-R`, `SISSA-R-A`).


In [ ]:
CONFIGS = ['SISSA-L-A', 'SISSA-L']   # LSTM + atenção / LSTM puro
for c in CONFIGS:
    print('='*25, 'TRAIN', c, '='*25)
    !python train.py --model_config config/{c}.yml


## 2. Avaliação + matriz de confusão (test.py dos autores)


In [ ]:
import glob, re, os
def best_weight(name):
    fs = glob.glob(f'results/weights/{name}/*.pt')
    f = lambda p: float((re.search(r'val-acc-([0-9.]+)', p) or [0,'0'])[1])
    return max(fs, key=f) if fs else None
for c in CONFIGS:
    w = best_weight(c)
    if w:
        print('='*25, 'TEST', c, '='*25)
        !python test.py --model_config config/{c}.yml --weights "{w}"


## 3. Resultados finais — métricas por classe + matriz + ROC

Recarrega o melhor modelo e calcula tudo sobre o conjunto de validação.


In [ ]:
MODEL = 'SISSA-L-A'   # modelo a detalhar
import numpy as np, glob, re, os, torch
import matplotlib.pyplot as plt, pandas as pd
from ruamel.yaml import YAML
from sklearn.metrics import (confusion_matrix, precision_recall_fscore_support,
                             accuracy_score, roc_curve, auc)
from sklearn.preprocessing import label_binarize
import sissautils, initialize
from models import SISSA_MODELS

initialize.init()
cfg = sissautils.update_model_config(model_config_path=f'config/{MODEL}.yml')
basic = YAML().load(open('config/basic.yml'))
names = basic['Preprocessor']['class_names']

model = SISSA_MODELS[cfg['Model']['type']](**cfg['Params']).cuda()
w = best_weight(MODEL); model.load_state_dict(torch.load(w)); model.eval()
print('pesos:', os.path.basename(w))

dl = sissautils.to_dataloader(cfg['Test']['test_dir'], 256, 0, False, 'Test')
probs, ys = [], []
with torch.no_grad():
    for x, y in dl:
        p = torch.softmax(model(x.cuda()), dim=1).cpu().numpy()
        probs.append(p); ys.append(y.numpy())
probs = np.concatenate(probs); y_true = np.concatenate(ys); y_pred = probs.argmax(1)


### 3a. Métricas por classe


In [ ]:
p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=range(len(names)), zero_division=0)
df = pd.DataFrame({'Precision': p, 'Recall': r, 'F1': f1, 'Support': sup}, index=names).round(3)
print(f'Modelo: {MODEL}  |  Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%')
df


### 3b. Matriz de confusão + curvas ROC (one-vs-rest)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
cm = confusion_matrix(y_true, y_pred, labels=range(len(names)))
im = ax1.imshow(cm, cmap='Blues')
ax1.set_xticks(range(len(names))); ax1.set_yticks(range(len(names)))
ax1.set_xticklabels(names, rotation=45, ha='right'); ax1.set_yticklabels(names)
ax1.set_xlabel('Predito'); ax1.set_ylabel('Verdadeiro'); ax1.set_title(f'Matriz de Confusão — {MODEL}')
for i in range(len(names)):
    for j in range(len(names)):
        ax1.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                 color='white' if cm[i, j] > cm.max()/2 else 'black')
Yb = label_binarize(y_true, classes=range(len(names)))
for i, cname in enumerate(names):
    fpr, tpr, _ = roc_curve(Yb[:, i], probs[:, i])
    ax2.plot(fpr, tpr, lw=1.5, label=f'{cname} (AUC={auc(fpr, tpr):.3f})')
ax2.plot([0, 1], [0, 1], 'k--', lw=0.7)
ax2.set_xlabel('Taxa de Falso Positivo'); ax2.set_ylabel('Taxa de Verdadeiro Positivo')
ax2.set_title('Curvas ROC por classe'); ax2.legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.show()


---
**Esperado (nossa reprodução):** SISSA-L-A e SISSA-L ~99,4% de val acc (artigo: 99,7%);
AUC perto de 1. Achado: a autoatenção (RSAB) quase não melhora vs. o LSTM puro.
